# Notebook 7: corrected group-correlated perturbation

Replaces Part F2 of Notebook 5. The earlier version scaled **Magnitude** by the packet-length factor *c*; by its definition (square root of a sum of mean lengths) it should scale by √*c*. It also scaled **Tot sum** by *c* only, although a sum of lengths also grows with the packet count. This version applies the transformation that matches each documented definition:

| Factor per record | Features | Scaling |
|---|---|---|
| length factor *c* | Header_Length, Min, Max, AVG, Std, Tot size, Radius | × *c* |
| | Magnitue (Magnitude) | × √*c* |
| | Covariance | × *c*² |
| count factor *k* | ack_count, syn_count, fin_count, rst_count, Number | × *k* |
| | Weight | × *k*² |
| both | Tot sum | × *c*·*k* |
| rate factor *r* | Rate | × *r* |
| none | indicators, Protocol Type, Duration, Variance, IAT | unchanged |

Run it after Notebook 4 has finished, in the `Python (iomt)` kernel, from the same folder as the other notebooks. It takes about 30 minutes. Send back `results_revision/correlated_v2_results.xlsx`.

In [1]:
# Load and run every code cell of Notebook 4 in this notebook, so its functions and data are available here.
# Its main loop skips runs already saved, so this only reloads data and rebuilds the perturbed test copies.
# (This replaces the Jupyter run command, which needs the nbformat package.)
import json as _json
_nb4 = _json.load(open("4_Supplementary_Analyses.ipynb", encoding="utf-8"))
for _cell in _nb4["cells"]:
    if _cell["cell_type"] == "code":
        _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]
        exec(compile(_src, "4_Supplementary_Analyses.ipynb", "exec"), globals())
print("Notebook 4 loaded.")

REV_OUT = "results_revision"; os.makedirs(REV_OUT, exist_ok=True)
P = "Proposed: NSGA-II (3-obj) + consensus"
assert len(state["runs"]) == N_RUNS, "Notebook 4 must finish all runs first"

def paired(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); n = len(d)
    h = stats.t.ppf(0.975, n - 1) * d.std(ddof=1) / np.sqrt(n)
    p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return {"mean diff": d.mean(), "95% CI low": d.mean() - h, "95% CI high": d.mean() + h,
            "runs favoring first": int((d > 0).sum()), "Wilcoxon p": p}
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); top = 0.0
    for r, i in enumerate(o):
        top = max(top, (m - r) * p[i]); adj[i] = min(1.0, top)
    return adj
print("Ready:", len(state["runs"]), "runs loaded")

Using checkpoint: results_revised\checkpoint.pkl
Windows-10-10.0.26300-SP0 | CPU: Intel64 Family 6 Model 142 Stepping 10, GenuineIntel | cores: 8 | Python 3.11.17 | scikit-learn 1.9.1 | pymoo 0.6.2 | numpy 2.4.6 | pandas 3.0.6
Runs in checkpoint: 10
Range-coded perturbed features: 20 | meaning-coded perturbed features: 19
Perturbed under range coding but not meaning coding: ['Protocol Type', 'Duration']
Perturbed under meaning coding but not range coding: ['Variance']
Perturbation sets: ['gauss_5', 'gauss_10', 'gauss_20', 'semantic_gauss_10', 'uniform', 'laplace', 'missing_10', 'missing_20', 'quantise']
Functions ready.
Saved progress found for runs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'max_abs_diff_clean_F1': np.float64(0.0), 'max_abs_diff_F1_noise10': np.float64(0.0), 'proposed_subset_reproduced_runs': 10, 'runs': 10}
{'mean difference': np.float64(-0.00495226344328078), '90% CI low': np.float64(-0.008142615247464569), '90% CI high': np.float64(-0.0017619116390969916), 'margin': 0.01, 'W

In [2]:
def grp(names): return [COL[c] for c in names if c in COL]
G_C = grp(["Header_Length", "Min", "Max", "AVG", "Std", "Tot size", "Radius"])
G_SQRTC = grp(["Magnitue", "Magnitude"]); G_C2 = grp(["Covariance"])
G_K = grp(["ack_count", "syn_count", "fin_count", "rst_count", "Number"]); G_K2 = grp(["Weight"])
G_CK = grp(["Tot sum"]); G_R = grp(["Rate"])
print("groups:", {k: [FEATURES_ALL[i] for i in v] for k, v in
      dict(c=G_C, sqrt_c=G_SQRTC, c2=G_C2, k=G_K, k2=G_K2, ck=G_CK, r=G_R).items()})

def correlated_v2(X, rng, s=0.10):
    Xn = X.astype("float64").copy(); n = len(X)
    c, k, r = (np.clip(1 + rng.normal(0, s, n), 0, None) for _ in range(3))
    Xn[:, G_C] *= c[:, None]; Xn[:, G_SQRTC] *= np.sqrt(c)[:, None]; Xn[:, G_C2] *= (c ** 2)[:, None]
    Xn[:, G_K] *= k[:, None]; Xn[:, G_K2] *= (k ** 2)[:, None]; Xn[:, G_CK] *= (c * k)[:, None]; Xn[:, G_R] *= r[:, None]
    return Xn.astype("float32")

PERT_C2 = [correlated_v2(X_test, np.random.default_rng(11000 + d)) for d in range(NOISE_DRAWS)]
iMin, iAVG, iMax = COL["Min"], COL["AVG"], COL["Max"]
broken = ((PERT_C2[0][:, iMin] > PERT_C2[0][:, iAVG] + 1e-6) | (PERT_C2[0][:, iAVG] > PERT_C2[0][:, iMax] + 1e-6)).mean()
print("share of rows breaking Min <= AVG <= Max:", round(float(broken), 6))

methods = ORIG + ["Random (k)", "MI bootstrap consensus (k)", "GA single-objective"]
rows = []
for run in range(N_RUNS):
    X_tr, y_tr = sample_train(run)
    for mth in methods:
        cols = [COL[f] for f in state["runs"][run]["subsets"][mth]]
        m = make_forest(FINAL_TREES, run).fit(X_tr[:, cols], y_tr)
        rows.append({"run": run, "method": mth, "n_features": len(cols),
                     "macro_F1": fast_macro_f1(yt_i, to_idx(m.predict(X_test[:, cols]))),
                     "F1_correlated_v2": float(np.mean([fast_macro_f1(yt_i, to_idx(m.predict(Xc[:, cols]))) for Xc in PERT_C2]))})
        del m
    gc.collect(); print(f"run {run + 1}/{N_RUNS} done", flush=True)
R7 = pd.DataFrame(rows)
g = R7.groupby("method")["F1_correlated_v2"]
M7 = pd.DataFrame({"mean": g.mean(), "95% CI half-width": stats.t.ppf(0.975, 9) * g.std(ddof=1) / np.sqrt(g.count())}).sort_values("mean", ascending=False)
w = R7.pivot(index="run", columns="method", values="F1_correlated_v2")
P7 = pd.DataFrame([{"vs": c, **paired(w[P], w[c])} for c in methods if c != P]); P7["Holm p"] = holm(P7["Wilcoxon p"].values)
print(M7.round(4)); print(P7.round(4).to_string(index=False))
with pd.ExcelWriter(os.path.join(REV_OUT, "correlated_v2_results.xlsx")) as xw:
    M7.to_excel(xw, sheet_name="Means"); P7.to_excel(xw, sheet_name="Paired", index=False)
    R7.to_excel(xw, sheet_name="Runs", index=False)
    pd.DataFrame({"quantity": ["share of rows breaking Min<=AVG<=Max"], "value": [float(broken)]}).to_excel(xw, sheet_name="Check", index=False)
print("Saved to", os.path.join(REV_OUT, "correlated_v2_results.xlsx"))

groups: {'c': ['Header_Length', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'Radius'], 'sqrt_c': ['Magnitue'], 'c2': ['Covariance'], 'k': ['ack_count', 'syn_count', 'fin_count', 'rst_count', 'Number'], 'k2': ['Weight'], 'ck': ['Tot sum'], 'r': ['Rate']}
share of rows breaking Min <= AVG <= Max: 0.0
run 1/10 done
run 2/10 done
run 3/10 done
run 4/10 done
run 5/10 done
run 6/10 done
run 7/10 done
run 8/10 done
run 9/10 done
run 10/10 done
                                         mean  95% CI half-width
method                                                          
All features + IAT (reference)         0.8503             0.0024
Proposed: NSGA-II (3-obj) + consensus  0.7150             0.0061
GA single-objective                    0.7058             0.0100
All features                           0.7053             0.0069
RFE (top-k)                            0.6962             0.0074
NSGA-II (3-obj)                        0.6902             0.0230
NSGA-II (2-obj)                        0.68